In [16]:
import duckdb
con = duckdb.connect("rfm.duckdb")

In [17]:
# Run once to load the CSV. Already saved in rfm.duckdb.
# con.sql("""
#     CREATE OR REPLACE TABLE sales AS
#     SELECT * FROM read_csv_auto('_data/online_retail.csv')
# """)
con.sql("DESCRIBE sales").df()

,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer ID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [18]:
con.sql("SELECT COUNT(*) AS total_rows FROM sales").df()

,total_rows
0,1067371


In [19]:
con.sql("SELECT MIN(InvoiceDate) AS first_date, MAX(InvoiceDate) AS last_date FROM sales").df()

,first_date,last_date
0,2009-12-01 07:45:00,2011-12-09 12:50:00


In [20]:
con.sql('SELECT COUNT(*) AS no_customer FROM sales WHERE "Customer ID" IS NULL').df()

,no_customer
0,243007


In [21]:
con.sql("SELECT COUNT(*) AS cancellations FROM sales WHERE Invoice LIKE 'C%'").df()

,cancellations
0,19494


In [22]:
con.sql("SELECT COUNT(*) AS bad_rows FROM sales WHERE Quantity <= 0 OR Price <= 0").df()

,bad_rows
0,25700


**Findings**
- 1,067,371 rows, Dec 2009 to Dec 2011
- 22.8% have no customer ID: removed, since RFM needs a customer
- 19,494 cancellation rows (invoice starts with C) and 25,700 rows with zero or negative quantity or price: removed

In [23]:
con.sql("""
SELECT Invoice, InvoiceDate, Quantity, Price, Quantity * Price AS line_total
FROM sales
WHERE "Customer ID" = 16446
ORDER BY InvoiceDate
""").df()

,Invoice,InvoiceDate,Quantity,Price,line_total
0,C525275,2010-10-04 16:38:00,-2,4.50,-9.00
1,553573,2011-05-18 09:52:00,1,1.65,1.65
2,553573,2011-05-18 09:52:00,1,1.25,1.25
3,581483,2011-12-09 09:15:00,80995,2.08,168469.60
4,C581484,2011-12-09 09:27:00,-80995,2.08,-168469.60


**Finding:** some orders were later fully cancelled (e.g. customer 16446: 168k order cancelled 12 minutes later). Removing only the "C" rows left the original order counting as spend, so I also remove orders with a matching cancellation.

In [24]:
con.sql("""
create or replace table clean_sales AS
select 
 invoice, 
 invoicedate, 
 CAST("Customer ID" AS INTEGER) AS customer_id,
 Quantity,
 Price, 
 Quantity * Price AS revenue
from sales s 
where s."customer ID" IS NOT NULL
  and s.Quantity > 0
  and s.Price > 0
  and s.Invoice not like 'C%'
and not exists (
    select 1 from sales c
    where c.invoice like 'C%' 
    and c."Customer ID" = s."Customer ID"
    and c.stockcode = s.stockcode
    and c.quantity = -s.quantity
    and c.invoicedate >= s.invoicedate)""")


CountBIGINT
795637


In [25]:
con.sql(""" 
select COUNT(*) AS rows_left, 
    COUNT(DISTINCT customer_id) AS customers,
    min(revenue) AS min_revenue
from clean_sales
  """).df()

,rows_left,customers,min_revenue
0,795637,5850,0.001


In [26]:
con.sql("""create or replace table rfm AS
select 
    customer_id,
    Round(sum(revenue), 2) AS monetary,
    count(distinct invoice) AS frequency,
    DATE_DIFF('day', max(invoicedate), TIMESTAMP '2011-12-10') AS recency
from clean_sales 
group by customer_id
""")

CountBIGINT
5850


In [27]:
con.sql(" select * from rfm order by monetary desc limit 10").df()

,customer_id,monetary,frequency,recency
0,18102,606963.25,145,1
1,14646,521424.26,146,2
2,14156,298274.92,149,10
3,14911,262975.46,374,2
4,17450,235864.75,49,9
5,13694,191445.63,141,4
6,17511,156230.55,58,3
7,12415,143350.72,26,25
8,16684,141193.97,53,5
9,15061,136969.28,126,4


In [28]:
con.sql("SELECT * FROM rfm WHERE customer_id = 16446").df()

,customer_id,monetary,frequency,recency
0,16446,2.9,1,206


In [29]:
con.sql("""
create or replace table rfm_scores AS
select
    customer_id ,recency, frequency , monetary,
    ntile(5) over (order by recency desc) AS r_score,
    ntile(5) over (order by frequency asc) AS f_score,
    ntile(5) over (order by monetary asc) AS m_score
from rfm
"""
    )

CountBIGINT
5850


In [30]:
con.sql("""
SELECT r_score, COUNT(*) AS customers, MIN(recency) AS min_days, MAX(recency) AS max_days
FROM rfm_scores
GROUP BY r_score
ORDER BY r_score
""").df()

,r_score,customers,min_days,max_days
0,1,1170,410,739
1,2,1170,188,410
2,3,1170,59,188
3,4,1170,20,59
4,5,1170,1,20


In [31]:
con.sql("""
CREATE OR REPLACE TABLE rfm_segments AS
SELECT *,
    CASE
        WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'Potential Loyalists'
        WHEN r_score = 3  AND f_score <= 2 THEN 'Needs Attention'
        WHEN r_score <= 2 AND f_score >= 4 THEN 'Cant Lose Them'
        WHEN r_score <= 2 AND f_score = 3  THEN 'At Risk'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'Hibernating'
        ELSE 'Other'
    END AS segment
FROM rfm_scores
""")

CountBIGINT
5850


In [32]:
con.sql("""
SELECT
    segment,
    COUNT(*) AS customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_customers,
    ROUND(SUM(monetary)) AS revenue,
    ROUND(100.0 * SUM(monetary) / SUM(SUM(monetary)) OVER (), 1) AS pct_revenue
FROM rfm_segments
GROUP BY segment
ORDER BY revenue DESC
""").df()

,segment,customers,pct_customers,revenue,pct_revenue
0,Champions,1451,24.8,11842668.0,70.1
1,Loyal Customers,1134,19.4,2447970.0,14.5
2,Cant Lose Them,420,7.2,1087065.0,6.4
3,Hibernating,1415,24.2,590305.0,3.5
4,At Risk,505,8.6,436977.0,2.6
5,Potential Loyalists,505,8.6,258139.0,1.5
6,Needs Attention,420,7.2,226362.0,1.3
